# Projeto (Seção 4) — solução comentada

Solução própria (aproveitando a sua ideia de `make_alias`), testes extras, revisão da sua tentativa (`08 - My_Project.py`) e comparação
com o instrutor. Enunciado em [`08_projeto`](08_projeto.ipynb).

## 1. Solução

A chave do exercício é perceber que **`alias` vale para as duas direções**, enquanto `validation_alias` vale só para a entrada. Os campos
que chegam com nome "estranho" mas devem sair em camelCase precisam de `validation_alias` (a saída continua usando o alias gerado).

In [1]:
from datetime import date
from enum import Enum
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_serializer
from pydantic.alias_generators import to_camel


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


def make_alias(field_name: str) -> str:
    """camelCase sem o '_' final: type_ -> type, series_name -> seriesName."""
    return to_camel(field_name).removesuffix("_")


class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=make_alias,           # resolve type_ <-> type sem Field explícito
    )

    manufacturer: str
    series_name: str
    type_: AutomobileType
    is_electric: bool = False
    manufactured_date: date = Field(validation_alias="completionDate")      # entra assim, sai manufacturedDate
    base_msrp_usd: float = Field(validation_alias="msrpUSD", serialization_alias="baseMSRPUSD")
    vin: str
    number_of_doors: int = Field(default=4, validation_alias="doors")       # entra assim, sai numberOfDoors
    registration_country: str | None = None
    license_plate: str | None = None

    @field_serializer("manufactured_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")

## 2. Testes do enunciado

In [2]:
data_json = '''{"manufacturer": "BMW", "seriesName": "M4", "type": "Convertible", "isElectric": false,
                "completionDate": "2023-01-01", "msrpUSD": 93300, "vin": "1234567890", "doors": 2,
                "registrationCountry": "France", "licensePlate": "AAA-BBB"}'''
expected_serialized_dict = {
    'manufacturer': 'BMW', 'series_name': 'M4', 'type_': AutomobileType.convertible, 'is_electric': False,
    'manufactured_date': date(2023, 1, 1), 'base_msrp_usd': 93300.0, 'vin': '1234567890',
    'number_of_doors': 2, 'registration_country': 'France', 'license_plate': 'AAA-BBB',
}
expected_serialized_dict_by_alias = {
    'manufacturer': 'BMW', 'seriesName': 'M4', 'type': AutomobileType.convertible, 'isElectric': False,
    'manufacturedDate': date(2023, 1, 1), 'baseMSRPUSD': 93300.0, 'vin': '1234567890',
    'numberOfDoors': 2, 'registrationCountry': 'France', 'licensePlate': 'AAA-BBB',
}
expected_serialized_json_by_alias = (
    '{"manufacturer":"BMW","seriesName":"M4","type":"Convertible",'
    '"isElectric":false,"manufacturedDate":"2023/01/01","baseMSRPUSD":93300.0,'
    '"vin":"1234567890","numberOfDoors":2,"registrationCountry":"France",'
    '"licensePlate":"AAA-BBB"}'
)

car = Automobile.model_validate_json(data_json)
assert car.model_dump() == expected_serialized_dict
assert car.model_dump(by_alias=True) == expected_serialized_dict_by_alias
assert car.model_dump_json(by_alias=True) == expected_serialized_json_by_alias
print(f"car.model_dump_json(by_alias=True): {car.model_dump_json(by_alias=True)}")
print("3 testes do enunciado: ok")

car.model_dump_json(by_alias=True): {"manufacturer":"BMW","seriesName":"M4","type":"Convertible","isElectric":false,"manufacturedDate":"2023/01/01","baseMSRPUSD":93300.0,"vin":"1234567890","numberOfDoors":2,"registrationCountry":"France","licensePlate":"AAA-BBB"}
3 testes do enunciado: ok


## 3. Testes extras

In [3]:
import json

def errors(data: dict) -> list[tuple]:
    try:
        Automobile.model_validate(data)
    except ValidationError as ex:
        return [(err["loc"][0], err["type"]) for err in ex.errors()]
    raise AssertionError("era esperado ValidationError")

source = json.loads(data_json)

# 1) JSON por NOME de campo (sem by_alias): o serializador de data também vale
print(f"car.model_dump_json(): {car.model_dump_json()}")
assert json.loads(car.model_dump_json())["manufactured_date"] == "2023/01/01"

# 2) validation_alias SUBSTITUI o alias gerado na entrada: o próprio formato de saída não é aceito de volta
round_trip = json.loads(car.model_dump_json(by_alias=True))
print(f"reenviar a saída por alias: {errors(round_trip)}")

# 3) por isso, nomes de campo também não entram (não há populate_by_name / validate_by_name)
print(f"entrada pelos nomes dos campos: {errors(car.model_dump())[:3]} ...")

# 4) defaults: doors ausente -> 4; anuláveis ausentes -> None
minimal = {k: v for k, v in source.items() if k not in ("doors", "registrationCountry", "licensePlate")}
m = Automobile.model_validate(minimal)
assert (m.number_of_doors, m.registration_country, m.license_plate) == (4, None, None)
print(f"m.number_of_doors (default): {m.number_of_doors}")
print("testes extras: ok")

car.model_dump_json(): {"manufacturer":"BMW","series_name":"M4","type_":"Convertible","is_electric":false,"manufactured_date":"2023/01/01","base_msrp_usd":93300.0,"vin":"1234567890","number_of_doors":2,"registration_country":"France","license_plate":"AAA-BBB"}
reenviar a saída por alias: [('completionDate', 'missing'), ('msrpUSD', 'missing'), ('manufacturedDate', 'extra_forbidden'), ('baseMSRPUSD', 'extra_forbidden'), ('numberOfDoors', 'extra_forbidden')]
entrada pelos nomes dos campos: [('seriesName', 'missing'), ('type', 'missing'), ('completionDate', 'missing')] ...
m.number_of_doors (default): 4
testes extras: ok


O teste 2 revela uma limitação do enunciado que vale conhecer: **o JSON que este modelo produz não pode ser lido de volta por ele mesmo**,
porque `completionDate`/`doors`/`msrpUSD` são os únicos nomes aceitos na entrada. Se o requisito fosse aceitar os dois formatos, a correção
é `AliasChoices`:

In [4]:
from pydantic import AliasChoices

class AutomobileRoundTrip(Automobile):
    manufactured_date: date = Field(validation_alias=AliasChoices("completionDate", "manufacturedDate"))
    base_msrp_usd: float = Field(validation_alias=AliasChoices("msrpUSD", "baseMSRPUSD"),
                                 serialization_alias="baseMSRPUSD")
    number_of_doors: int = Field(default=4, validation_alias=AliasChoices("doors", "numberOfDoors"))

back = AutomobileRoundTrip.model_validate(car.model_dump(by_alias=True, mode="python"))
print(f"ida e volta: {back.model_dump() == expected_serialized_dict}")

ida e volta: True


## 4. Revisão da sua tentativa (`08 - My_Project.py`)

**O que está ótimo:** o `make_alias` (camelCase + `removesuffix("_")`) é mais elegante que o `Field(alias="type")` do instrutor — resolve
`type_` e qualquer futuro `id_`/`list_` de uma vez. Adotei aqui. O `base_msrp_usd` com `validation_alias` + `serialization_alias` também está correto.

**O que quebra:** discordo de usar `alias=` em `manufactured_date` e `number_of_doors` porque `alias` define o nome nas **duas** direções —
a serialização por alias passa a sair como `completionDate` e `doors`, e o enunciado pede `manufacturedDate`/`numberOfDoors`. O que eu faria
no lugar: `validation_alias=`. O risco do seu caminho: o teste `model_dump(by_alias=True)` falha. Também faltou o `@field_serializer` do
formato `YYYY/MM/DD`. Rodando a sua classe contra os três testes:

In [5]:
class AutomobileAttempt(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True, validate_default=True,
                              validate_assignment=True, alias_generator=make_alias)
    manufacturer: str
    series_name: str
    type_: AutomobileType
    is_electric: bool = False
    manufactured_date: date = Field(alias="completionDate", )
    base_msrp_usd: float = Field(validation_alias="msrpUSD", serialization_alias="baseMSRPUSD")
    vin: str
    number_of_doors: int = Field(alias="doors", default=4)
    registration_country: str | None = None
    license_plate: str | None = None

attempt = AutomobileAttempt.model_validate_json(data_json)
results = {
    "model_dump()": attempt.model_dump() == expected_serialized_dict,
    "model_dump(by_alias=True)": attempt.model_dump(by_alias=True) == expected_serialized_dict_by_alias,
    "model_dump_json(by_alias=True)": attempt.model_dump_json(by_alias=True) == expected_serialized_json_by_alias,
}
for test, ok in results.items():
    print(f"{test}: {'ok' if ok else 'FALHA'}")
print(f"chaves divergentes: {set(attempt.model_dump(by_alias=True)) ^ set(expected_serialized_dict_by_alias)}")

model_dump(): ok
model_dump(by_alias=True): FALHA
model_dump_json(by_alias=True): FALHA
chaves divergentes: {'manufacturedDate', 'numberOfDoors', 'doors', 'completionDate'}


## 5. Comparação com a solução do instrutor

| ponto | instrutor | esta solução |
|---|---|---|
| gerador de alias | `to_camel` + `Field(alias="type")` | `make_alias` (sua ideia): `type_` sai sem `Field` |
| entrada renomeada | `validation_alias` | `validation_alias` (idem) |
| data no JSON | `field_serializer(..., when_used="json-unless-none")` | idem |
| testes | os 3 do enunciado | + JSON por nome, ida e volta, entrada por nome, defaults |
| achado | — | a saída do modelo não é aceita como entrada; correção com `AliasChoices` |